In [ ]:
import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine

#### Conexion con las bases de datos

In [ ]:
config = {
    'host': 'localhost',       
    'port': 
    'database': 'mensajeria',  
    'user': 'postgres',            
    'password':
}

In [ ]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

    config_co = config['SOURCE_DB']
    config_etl = config['ETL_PRO']

url_co = f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:{config_co['port']}/{config_co['dbname']}"
url_etl = f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:{config_etl['port']}/{config_etl['dbname']}"


engine_origen = create_engine(url_co)
engine_destino = create_engine(url_etl)

In [ ]:
cliente = pd.read_sql_table("cliente", engine_origen)
cliente.head()

,cliente_id,nit_cliente,nombre,email,direccion,telefono,nombre_contacto,ciudad_id,tipo_cliente_id,activo,coordinador_id,sector
0,1,25,Cliente 2,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,S
1,2,123,Cliente 1,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,2.0,industrial
2,6,24390-3,CLINICA DEPORTIVA DEL SUR,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,1.0,salud
3,19,8301821,HOSPITAL ORTOPEDICO DE COLOMBIA,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
4,8,5017350-8,CLINICA NEFROLOGOS DE CALI,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud


#### Renombrar columnas

In [ ]:
cliente.rename(columns={
    'cliente_id': 'id_cliente',
    'nombre': 'nombre_cliente',
    'tipo_cliente_id': 'tipo_cliente'
}, inplace=True)


In [ ]:
columnas_necesarias = ["id_cliente", "nombre_cliente", "tipo_cliente", "email", "telefono", "sector"]
dim_cliente = cliente[columnas_necesarias].copy()
dim_cliente.head()

,id_cliente,nombre_cliente,tipo_cliente,email,telefono,sector
0,1,Cliente 2,1,algo.com,327-00000,S
1,2,Cliente 1,1,algo.com,327-00000,industrial
2,6,CLINICA DEPORTIVA DEL SUR,1,algo.com,327-00000,salud
3,19,HOSPITAL ORTOPEDICO DE COLOMBIA,1,algo.com,327-00000,salud
4,8,CLINICA NEFROLOGOS DE CALI,1,algo.com,327-00000,salud


#### Carga de datos

In [ ]:
dim_cliente.to_sql('dim_cliente',con=engine_destino,if_exists='replace',index_label='key_dim_cliente')

27